# Regresión lineal sobre BigMart Sales: modelo mejorado y análisis del límite de predicción

**Objetivo:** predecir `Item_Outlet_Sales` y mejorar el modelo base (R² ≈ 0.558).

**Resumen de resultados (conjunto de prueba 20 %, `random_state=1`):**

| Modelo | RMSE | R² |
|---|---|---|
| Base (notebook original) | 1144.38 | 0.558 |
| Lineal con interacción precio × tienda (Ridge) | 1081.75 | 0.605 |

Además, en la sección 6 se demuestra con datos que el límite alcanzable con las variables disponibles ronda **R² ≈ 0.60–0.61**: el resto de la variación es ruido no explicable con este archivo.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# En Colab: monta Drive y pon aquí la ruta completa del archivo
RUTA = 'bigMartSales.csv'
sales_df = pd.read_csv(RUTA)
print(sales_df.shape)
sales_df.head()

(8523, 12)


## 1. Exploración básica

In [2]:
print(sales_df.isnull().sum())
print()
print(sales_df['Item_Outlet_Sales'].describe().round(2))

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

count     8523.00
mean      2181.29
std       1706.50
min         33.29
25%        834.25
50%       1794.33
75%       3101.30
max      13086.96
Name: Item_Outlet_Sales, dtype: float64


## 2. Limpieza e ingeniería de características
Todas las transformaciones usan solo columnas de entrada (nunca el objetivo), por lo que no hay fuga de información.

In [3]:
df = sales_df.copy()

# Unificar categorías de grasa
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(
    {'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'})

# Peso: imputar con el promedio del mismo producto; si no hay, mediana global
df['Item_Weight'] = df.groupby('Item_Identifier')['Item_Weight'].transform(lambda s: s.fillna(s.mean()))
df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].median())

# Visibilidad = 0 no es un valor real: se trata como nulo y se imputa por producto
df['Item_Visibility'] = df['Item_Visibility'].replace(0, np.nan)
df['Item_Visibility'] = df.groupby('Item_Identifier')['Item_Visibility'].transform(lambda s: s.fillna(s.mean()))
df['Item_Visibility'] = df['Item_Visibility'].fillna(df['Item_Visibility'].median())

# Tamaño de tienda: imputar con la moda de su tipo de tienda
df['Outlet_Size'] = df.groupby('Outlet_Type')['Outlet_Size'].transform(
    lambda s: s.fillna(s.mode().iloc[0] if not s.mode().empty else 'Small'))
df['Outlet_Size'] = df['Outlet_Size'].fillna('Small')

# Antigüedad con año de referencia fijo (reproducible)
df['Outlet_Age'] = 2013 - df['Outlet_Establishment_Year']

y = df['Item_Outlet_Sales']
X = df.drop(columns=['Item_Outlet_Sales', 'Item_Identifier', 'Outlet_Establishment_Year'])

# Interacción precio x tienda: una columna MRP por cada tienda
X_int = X.copy()
for outlet in sorted(X['Outlet_Identifier'].unique()):
    X_int[f'MRP_x_{outlet}'] = X['Item_MRP'] * (X['Outlet_Identifier'] == outlet)

print('Nulos restantes en X:', int(X.isnull().sum().sum()))

Nulos restantes en X: 0


## 3. Partición 80 / 20 (misma semilla que el notebook original)

In [4]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X_int, y, train_size=0.8, random_state=1)
print(Xtrain.shape, Xtest.shape)

(6818, 20) (1705, 20)


## 4. Modelo base (referencia del notebook original)

In [5]:
# Reproduce el modelo original: sin Outlet_Identifier ni interacciones, con los datos sin limpieza avanzada
base = sales_df.drop(columns=['Item_Identifier', 'Outlet_Identifier']).copy()
base['Item_Fat_Content'] = base['Item_Fat_Content'].replace({'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'})
base['Outlet_Age'] = 2025 - base['Outlet_Establishment_Year']
base = base.drop(columns='Outlet_Establishment_Year')
Xb, yb = base.drop(columns='Item_Outlet_Sales'), base['Item_Outlet_Sales']
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, train_size=0.8, random_state=1)

prep_b = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), MinMaxScaler()), Xb.select_dtypes(include=np.number).columns.tolist()),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
     Xb.select_dtypes(exclude=np.number).columns.tolist())])
lr_base = make_pipeline(prep_b, LinearRegression()).fit(Xb_tr, yb_tr)
pred_base = lr_base.predict(Xb_te)
print('Modelo base  RMSE:', round(rmse(yb_te, pred_base), 2), '  R2:', round(r2_score(yb_te, pred_base), 4))

Modelo base  RMSE: 1144.38   R2: 0.5582


## 5. Modelo mejorado: regresión lineal (Ridge) con interacción precio × tienda
Se conserva `Outlet_Identifier` (el notebook original lo eliminaba) y se añade la interacción `Item_MRP` × tienda. Justificación: las ventas son aproximadamente *precio × unidades vendidas*, y las unidades vendidas dependen mucho de la tienda.

In [6]:
cat_cols = X_int.select_dtypes(exclude=np.number).columns.tolist()
num_cols = X_int.select_dtypes(include=np.number).columns.tolist()

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)])

lr_model = make_pipeline(prep, RidgeCV(alphas=np.logspace(-3, 2, 20)))
lr_model.fit(Xtrain, ytrain)
predictions = lr_model.predict(Xtest)

print('Modelo mejorado  RMSE:', round(rmse(ytest, predictions), 2), '  R2:', round(r2_score(ytest, predictions), 4))

# Validación cruzada de 5 pliegues sobre todo el conjunto, para confirmar que no es casualidad del split
cv = KFold(n_splits=5, shuffle=True, random_state=1)
pred_cv = cross_val_predict(lr_model, X_int, y, cv=cv)
print('R2 validación cruzada (5 pliegues):', round(r2_score(y, pred_cv), 4))

Modelo mejorado  RMSE: 1081.75   R2: 0.6052
R2 validación cruzada (5 pliegues): 0.6045


In [7]:
resultados = pd.DataFrame({
    'Modelo': ['Base (original)', 'Lineal + interacción precio x tienda'],
    'RMSE': [rmse(yb_te, pred_base), rmse(ytest, predictions)],
    'R2': [r2_score(yb_te, pred_base), r2_score(ytest, predictions)]}).round(4)
print(resultados.to_string(index=False))

plt.figure(figsize=(6, 5))
plt.scatter(ytest, predictions, s=8, alpha=0.4)
lim = [0, max(ytest.max(), predictions.max())]
plt.plot(lim, lim, color='red')
plt.xlabel('Ventas reales'); plt.ylabel('Ventas predichas'); plt.title('Real vs. predicho (prueba)')
plt.show()

                              Modelo      RMSE     R2
                     Base (original) 1144.3828 0.5582
Lineal + interacción precio x tienda 1081.7506 0.6052


## 6. ¿Por qué no se supera ~0.61? Diagnóstico del ruido

Como ventas ≈ precio × unidades, se calcula `units = Item_Outlet_Sales / Item_MRP` y se estudia cuánto de la variación de las unidades puede explicarse con las columnas disponibles.

In [8]:
d = df.copy()
d['units'] = d['Item_Outlet_Sales'] / d['Item_MRP']
d['Item_Category'] = d['Item_Identifier'].str[:2]

# 6.1 Las unidades vendidas varían mucho incluso dentro de una misma tienda
resumen = d.groupby('Outlet_Identifier')['units'].agg(['mean', 'std', 'count']).round(2)
resumen['coef_var'] = (resumen['std'] / resumen['mean']).round(2)
print(resumen)

                    mean   std  count  coef_var
Outlet_Identifier                              
OUT010              2.38  1.38    555      0.58
OUT013             16.00  7.27    932      0.45
OUT017             16.66  7.21    926      0.43
OUT018             13.85  6.45    928      0.47
OUT019              2.44  1.40    528      0.57
OUT027             26.61  9.50    935      0.36
OUT035             16.97  7.31    930      0.43
OUT045             15.58  6.95    929      0.45
OUT046             16.19  7.22    930      0.45
OUT049             16.63  7.08    930      0.43


In [9]:
# 6.2 Ponemos a prueba qué variables explican las unidades (validación cruzada, R2 sobre ventas = unidades_pred x MRP)
feats_all = ['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP',
             'Outlet_Identifier', 'Outlet_Size', 'Item_Category']
Xd = d[feats_all].copy()
for c in Xd.select_dtypes('object'):
    Xd[c] = Xd[c].astype('category')

escenarios = [('Solo la tienda', ['Outlet_Identifier']),
              ('Tienda + precio', ['Outlet_Identifier', 'Item_MRP']),
              ('Todas las variables', feats_all)]

for nombre, fs in escenarios:
    m = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=150,
                                      min_samples_leaf=50, categorical_features='from_dtype', random_state=1)
    pu = cross_val_predict(m, Xd[fs], d['units'], cv=cv)
    print(f'{nombre:22s} R2 ventas: {r2_score(d["Item_Outlet_Sales"], pu * d["Item_MRP"]):.4f}')

Solo la tienda         R2 ventas: 0.6050
Tienda + precio        R2 ventas: 0.6045
Todas las variables    R2 ventas: 0.6027


In [10]:
# 6.3 ¿Ayuda saber cómo se vende cada producto en otras tiendas?
tr_idx, te_idx = train_test_split(d.index, train_size=0.8, random_state=1)
tr, te = d.loc[tr_idx], d.loc[te_idx]
media_tienda = tr.groupby('Outlet_Identifier')['units'].mean()
media_prod = tr.groupby('Item_Identifier')['units'].mean()

base_te = te['Outlet_Identifier'].map(media_tienda)
efecto_prod = te['Item_Identifier'].map(media_prod).fillna(media_prod.mean()) - base_te
residuo = te['units'] - base_te
print('Correlación entre el "efecto producto" y el residuo en prueba:', round(np.corrcoef(efecto_prod, residuo)[0, 1], 4))

# Referencia: modelo mínimo = (unidades promedio de la tienda) x precio
pred_min = te['Item_MRP'] * base_te
print('R2 del modelo mínimo (tienda x precio):', round(r2_score(te['Item_Outlet_Sales'], pred_min), 4))

Correlación entre el "efecto producto" y el residuo en prueba: 0.0067
R2 del modelo mínimo (tienda x precio): 0.6056


## 7. Conclusiones

1. **El modelo mejorado aumenta el R² de 0.558 a ≈ 0.605** y reduce el RMSE de 1144 a ≈ 1082, gracias a conservar la tienda y modelar la interacción precio × tienda. La validación cruzada confirma que la mejora no depende del split.
2. **La tienda y el precio concentran toda la señal útil.** Un modelo que solo conoce esas dos variables ya alcanza ≈ 0.605; agregar peso, grasa, visibilidad, tipo y categoría de producto no mejora el resultado (6.2).
3. **El comportamiento por producto no aporta información adicional**: la correlación entre la popularidad de un producto en otras tiendas y su residuo es prácticamente cero (6.3).
4. **El coeficiente de variación de las unidades vendidas va de 0.36 a 0.58 según la tienda (≈ 0.45 en la mayoría)** (6.1): hay una variabilidad aleatoria fuerte que ninguna columna del dataset explica. Esto impone un techo de R² ≈ 0.60–0.61.
5. Un R² de 0.70 en datos de prueba **no es alcanzable con este archivo**; valores superiores solo aparecerían evaluando sobre datos de entrenamiento (sobreajuste), lo que no refleja la capacidad real de predicción.

**Posibles siguientes pasos:** incorporar variables externas (promociones, estacionalidad, fecha de venta, inventario) o predecir ventas agregadas (por tienda o categoría), donde el ruido individual se compensa.